# TwinGuard-SD-IoT: GPU training for the revision

Trains, on a Kaggle GPU, the models of the revision that are too slow on a laptop CPU:

1. **Action-conditioned twins on the simulator** (dataset topology): graph-attention twin and per-link twin, seeds 0 and 1.
2. **Twins of the two new topologies** (NSFNET, 14 routers / 21 links; two-tier IoT access network, 12 routers / 22 links): graph-attention twin and per-link twin, seeds 0 and 1.
3. **Equal-capacity comparison on the public dataset**: per-link GRU with the GNN's configuration (78,070 parameters) and GNN twin with the GRU's configuration (34,166 parameters), seeds 0-4.

Architectures, hyperparameters, losses, and training protocol (at most 150 epochs, early stopping with patience 15, as for the published models) are those of `twinguard/model.py` (bundled as `twinguard_model.py`); only the device changes. Checkpoints are saved on the CPU in the format the repository loads.

**How to run**
1. *Add data* -> upload `twinguard_kaggle_inputs.zip` as a new dataset (any name).
2. *Settings* -> Accelerator: **GPU T4 x1** (or P100).
3. *Run all* (about 45-75 min). The notebook is resumable: finished checkpoints are skipped.
4. Download `/kaggle/working/twinguard_outputs.zip` (Output panel) and send it back.

In [ ]:
import copy, glob, json, os, sys, time, importlib.util, shutil
from pathlib import Path
import numpy as np, torch, torch.nn as nn

# locate the uploaded bundle wherever Kaggle mounted it
hits = glob.glob('/kaggle/input/**/best_hyperparameters.json', recursive=True)
assert hits, 'Add twinguard_kaggle_inputs.zip as a dataset (Add data -> Upload)'
IN = Path(hits[0]).parent
OUT = Path('/kaggle/working/twinguard_outputs'); OUT.mkdir(parents=True, exist_ok=True)
spec = importlib.util.spec_from_file_location('twinguard_model', IN / 'twinguard_model.py')
M = importlib.util.module_from_spec(spec); spec.loader.exec_module(M)
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HP = json.load(open(IN / 'best_hyperparameters.json'))
print('input:', IN); print('device:', DEV, torch.cuda.get_device_name(0) if DEV.type == 'cuda' else '')
print(sorted(p.name for p in IN.iterdir()))

In [ ]:
def train(cls, hp, fixed, fit, val, adj, seed, log):
    """twinguard.model.train on a GPU: AdamW, LR halving on plateau, early stopping on the validation loss."""
    torch.manual_seed(seed); np.random.seed(seed); gen = torch.Generator().manual_seed(seed)
    model = cls(fit['X'].shape[-1], hp, adj).to(DEV)
    T = lambda d: {k: torch.as_tensor(d[k]).to(DEV) for k in ('X', 'Y', 'Ym', 'V')}
    fit_t, val_t = T(fit), T(val)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=fixed['lr_factor'], patience=fixed['lr_patience'])
    best, best_state, best_ep, wait, hist, t0 = np.inf, copy.deepcopy(model.state_dict()), 0, 0, [], time.time()
    n = len(fit_t['X'])
    for ep in range(fixed['max_epochs']):
        model.train(); tl, nb = 0.0, 0
        order = torch.randperm(n, generator=gen)
        for i in range(0, n, hp['batch']):
            j = order[i:i + hp['batch']].to(DEV)
            q, lg = model(fit_t['X'][j]); l, _, _ = M.loss_fn(q, lg, fit_t['Y'][j], fit_t['Ym'][j], fit_t['V'][j])
            opt.zero_grad(set_to_none=True); l.backward()
            nn.utils.clip_grad_norm_(model.parameters(), fixed['clip']); opt.step()
            tl += l.item(); nb += 1
        model.eval(); vt, vn = 0.0, 0
        with torch.no_grad():
            for i in range(0, len(val_t['X']), 1024):
                sl = slice(i, i + 1024)
                q, lg = model(val_t['X'][sl]); l, _, _ = M.loss_fn(q, lg, val_t['Y'][sl], val_t['Ym'][sl], val_t['V'][sl])
                vt += l.item() * len(q); vn += len(q)
        vl = vt / vn; hist.append((tl / nb, vl)); sched.step(vl)
        log(f'    epoch {ep + 1}: train {tl / nb:.4f}  val {vl:.4f}')
        if vl < best - 1e-4:
            best, best_ep, wait = vl, ep + 1, 0; best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= fixed['patience']:
                break
    state = {k: v.detach().cpu() for k, v in best_state.items()}
    return state, {'best_val_loss': float(best), 'best_epoch': best_ep, 'epochs_run': len(hist), 'hist': hist,
                   'seconds': time.time() - t0, 'device': str(DEV)}


def run_job(path, cls_name, hp, fixed, fit, val, adj, seed, extra=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        print('exists:', path.relative_to(OUT)); return
    logf = open(path.with_suffix('.log'), 'w')
    state, info = train(M.MODELS[cls_name], hp, fixed, fit, val, adj, seed, lambda m: (logf.write(m + '\n'), logf.flush()))
    torch.save({'state': state, 'hp': hp, **(extra or {}), **info}, path)
    print(f"{path.relative_to(OUT)}: best epoch {info['best_epoch']}/{info['epochs_run']}, "
          f"val loss {info['best_val_loss']:.4f}, {info['seconds'] / 60:.1f} min", flush=True)

## 1-2. Simulator twins (dataset topology, NSFNET, two-tier: graph-attention and per-link twins)

In [ ]:
SIM_FIXED = HP['fixed']        # same protocol as the public dataset: at most 150 epochs, patience 15, LR halving after 5
JOBS = [(t, n) for t in ('dataset', 'nsfnet', 'two_tier') for n in ('GNNTwin', 'LinkGRU')]
for topo, name in JOBS:
    d = IN / f'sim_{topo}'
    load = lambda s: {k: v for k, v in np.load(d / f'{s}.npz').items() if k in ('X', 'Y', 'Ym', 'V')}
    fit, val, adj = load('train'), load('val'), np.load(d / 'adjacency.npy')
    sub = 'sim' if topo == 'dataset' else f'sim_{topo}'
    for seed in (0, 1):
        run_job(OUT / sub / 'models' / f'{name}_s{seed}.pt', name, HP['best'][name], SIM_FIXED, fit, val, adj, seed,
                {'fixed': SIM_FIXED})

## 3. Equal-capacity comparison on the public dataset

In [ ]:
pipe = np.load(IN / 'real' / 'graph_pipeline.npz')

def windows(split):
    X, Y, Ym, V, k = [], [], [], [], 0
    while f'{split}_t_end_{k}' in pipe.files:
        Xs, idx = pipe[f'{split}_X_{k}'], pipe[f'{split}_t_end_{k}']; W = int(pipe['W'])
        X.append(np.stack([Xs[t - W + 1:t + 1] for t in idx]))
        Y.append(pipe[f'{split}_Y_{k}'].transpose(0, 2, 1, 3)); Ym.append(pipe[f'{split}_Ym_{k}'].transpose(0, 2, 1))
        V.append(pipe[f'{split}_vio_{k}'].transpose(0, 2, 1)); k += 1
    return {'X': np.concatenate(X), 'Y': np.concatenate(Y), 'Ym': np.concatenate(Ym), 'V': np.concatenate(V)}

tr = windows('train'); cut = int(0.85 * len(tr['X']))          # last 15% of the training period: validation
fit = {k: v[:cut] for k, v in tr.items()}; val = {k: v[cut:] for k, v in tr.items()}
print('fit', fit['X'].shape, 'val', val['X'].shape)
for arch, cfg in [('LinkGRU', 'GNNTwin'), ('GNNTwin', 'LinkGRU')]:
    for seed in range(5):
        run_job(OUT / 'matched_capacity' / 'models' / f'{arch}_{cfg}config_s{seed}.pt', arch, HP['best'][cfg],
                HP['fixed'], fit, val, pipe['A'], seed)

## Summary and download

In [ ]:
rows = []
for p in sorted(OUT.rglob('*.pt')):
    ck = torch.load(p, map_location='cpu', weights_only=False)
    rows.append((str(p.relative_to(OUT)), ck['best_epoch'], ck['epochs_run'], round(ck['best_val_loss'], 4), round(ck['seconds'] / 60, 1)))
for r in rows:
    print('%-55s best %3d / %3d  val %.4f  %5.1f min' % r)
shutil.make_archive('/kaggle/working/twinguard_outputs', 'zip', OUT)
print('download /kaggle/working/twinguard_outputs.zip  (', len(rows), 'checkpoints )')